## load packages

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split, KFold
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

pd.set_option('display.max_columns', None)

In [2]:
import os

print(os.getcwd())

/home/jovyan/sp25-student/grad-proj/nlp-chatbot-analysis


In [3]:
import warnings
warnings.filterwarnings('ignore')

## load data

In [4]:
dat = pd.read_csv('complete_data_for_tasks_deciphered.csv')

print(dat.shape)
dat.head(5)

(25256, 16)


,prompt,topic_modeling_1,score_reason_1,score_value_1,topic_modeling_2,score_reason_2,score_value_2,topic_modeling_3,score_reason_3,score_value_3,topic_modeling_1_cluster,topic_modeling_2_cluster,topic_modeling_3_cluster,model_a,model_b,winner
0,What is the difference between OpenCL and CUDA?,Technical Comparison,This prompt requires the AI to accurately comp...,9,Software Comparison,This prompt assesses the AI's factual accuracy...,8,"Comparison, Technology",This prompt requires the AI to demonstrate kno...,9,"Comparison, Time","Technology, Engineering","Comparison, Time",chatglm-6b,koala-13b,model_b
1,Why did my parent not invite me to their wedding?,"Reasoning, Emotion",This prompt requires the AI to understand huma...,9,"Emotions, Relationships",This prompt involves understanding complex hum...,8,"Reasoning, Emotional",This prompt challenges the AI to infer motives...,8,"Emotional, Psychology","Emotional, Psychology","Emotional, Psychology",oasst-pythia-12b,alpaca-13b,tie
2,"Fuji vs. Nikon, which is better?",Camera comparison,This prompt does not require problem-solving s...,2,Comparative Analysis,This prompt assesses the AI's ability to analy...,6,Photography comparison,This prompt is subjective and does not provide...,2,"Identification, Classification","Comparison, Time","Identification, Classification",koala-13b,oasst-pythia-12b,model_b
3,How to build an arena for chatbots?,Chatbot Arena,This prompt requires problem-solving skills an...,8,Chatbot Arena,This prompt requires the AI to engage in probl...,8,Chatbot Arena,This prompt requires problem-solving skills an...,8,"Communication, Chatbot","Communication, Chatbot","Communication, Chatbot",vicuna-13b,oasst-pythia-12b,model_b
4,When is it today?,Time Query,This prompt is very straightforward and does n...,2,Date Inquiry,This prompt is very straightforward and does n...,2,Time-based Inquiry,This prompt is too straightforward and simply ...,2,"Comparison, Time","Comparison, Time","Comparison, Time",vicuna-13b,koala-13b,model_a


In [5]:
prompt_embeddings = np.load('prompt_embeddings.npy')
response_a_embeddings = np.load('response_a_embeddings.npy')
response_b_embeddings = np.load('response_b_embeddings.npy')

print(prompt_embeddings.shape)
print(response_a_embeddings.shape)
print(response_b_embeddings.shape)

(25256, 256)
(25256, 256)
(25256, 256)


## feature engineering

In [6]:
# ground truth
dat['winner'].replace(inplace=True, to_replace={'model_a':0,
                                                'model_b':1,
                                                'tie (bothbad)':2,
                                                'tie':3})

dat.head()

,prompt,topic_modeling_1,score_reason_1,score_value_1,topic_modeling_2,score_reason_2,score_value_2,topic_modeling_3,score_reason_3,score_value_3,topic_modeling_1_cluster,topic_modeling_2_cluster,topic_modeling_3_cluster,model_a,model_b,winner
0,What is the difference between OpenCL and CUDA?,Technical Comparison,This prompt requires the AI to accurately comp...,9,Software Comparison,This prompt assesses the AI's factual accuracy...,8,"Comparison, Technology",This prompt requires the AI to demonstrate kno...,9,"Comparison, Time","Technology, Engineering","Comparison, Time",chatglm-6b,koala-13b,1
1,Why did my parent not invite me to their wedding?,"Reasoning, Emotion",This prompt requires the AI to understand huma...,9,"Emotions, Relationships",This prompt involves understanding complex hum...,8,"Reasoning, Emotional",This prompt challenges the AI to infer motives...,8,"Emotional, Psychology","Emotional, Psychology","Emotional, Psychology",oasst-pythia-12b,alpaca-13b,3
2,"Fuji vs. Nikon, which is better?",Camera comparison,This prompt does not require problem-solving s...,2,Comparative Analysis,This prompt assesses the AI's ability to analy...,6,Photography comparison,This prompt is subjective and does not provide...,2,"Identification, Classification","Comparison, Time","Identification, Classification",koala-13b,oasst-pythia-12b,1
3,How to build an arena for chatbots?,Chatbot Arena,This prompt requires problem-solving skills an...,8,Chatbot Arena,This prompt requires the AI to engage in probl...,8,Chatbot Arena,This prompt requires problem-solving skills an...,8,"Communication, Chatbot","Communication, Chatbot","Communication, Chatbot",vicuna-13b,oasst-pythia-12b,1
4,When is it today?,Time Query,This prompt is very straightforward and does n...,2,Date Inquiry,This prompt is very straightforward and does n...,2,Time-based Inquiry,This prompt is too straightforward and simply ...,2,"Comparison, Time","Comparison, Time","Comparison, Time",vicuna-13b,koala-13b,0


In [7]:
# one hot encode categorical features
enc = OneHotEncoder()
enc.fit(dat[['model_a','model_b','topic_modeling_1_cluster']])
one_hot_encoded_df = pd.DataFrame(
    enc.transform(dat[['model_a','model_b','topic_modeling_1_cluster']]).toarray(),
    columns = enc.get_feature_names_out()
)
print(one_hot_encoded_df.shape)
one_hot_encoded_df.head()

(25256, 70)


,model_a_RWKV-4-Raven-14B,model_a_alpaca-13b,model_a_chatglm-6b,model_a_claude-instant-v1,model_a_claude-v1,model_a_dolly-v2-12b,model_a_fastchat-t5-3b,model_a_gpt-3.5-turbo,model_a_gpt-4,model_a_gpt4all-13b-snoozy,model_a_guanaco-33b,model_a_koala-13b,model_a_llama-13b,model_a_mpt-7b-chat,model_a_oasst-pythia-12b,model_a_palm-2,model_a_stablelm-tuned-alpha-7b,model_a_vicuna-13b,model_a_vicuna-7b,model_a_wizardlm-13b,model_b_RWKV-4-Raven-14B,model_b_alpaca-13b,model_b_chatglm-6b,model_b_claude-instant-v1,model_b_claude-v1,model_b_dolly-v2-12b,model_b_fastchat-t5-3b,model_b_gpt-3.5-turbo,model_b_gpt-4,model_b_gpt4all-13b-snoozy,model_b_guanaco-33b,model_b_koala-13b,model_b_llama-13b,model_b_mpt-7b-chat,model_b_oasst-pythia-12b,model_b_palm-2,model_b_stablelm-tuned-alpha-7b,model_b_vicuna-13b,model_b_vicuna-7b,model_b_wizardlm-13b,"topic_modeling_1_cluster_AI, Task","topic_modeling_1_cluster_Communication, Chatbot","topic_modeling_1_cluster_Comparison, Time","topic_modeling_1_cluster_Counting, Sorting","topic_modeling_1_cluster_Creativity, Creative","topic_modeling_1_cluster_Data, Database","topic_modeling_1_cluster_Emotional, Psychology","topic_modeling_1_cluster_Ethics, Legal","topic_modeling_1_cluster_Evaluation, Analysis","topic_modeling_1_cluster_Factual, Philosophy","topic_modeling_1_cluster_Financial, Finance","topic_modeling_1_cluster_Food, Recipe","topic_modeling_1_cluster_Game, Gaming","topic_modeling_1_cluster_Geography, Astronomy","topic_modeling_1_cluster_History, Historical","topic_modeling_1_cluster_Identification, Classification","topic_modeling_1_cluster_Language, Text","topic_modeling_1_cluster_Logic, Reasoning","topic_modeling_1_cluster_Mathematics, Mathematical","topic_modeling_1_cluster_Medical, Biology","topic_modeling_1_cluster_Physics, Chemistry","topic_modeling_1_cluster_Poetry, Music","topic_modeling_1_cluster_Problem-solving, Optimization","topic_modeling_1_cluster_Programming, Development","topic_modeling_1_cluster_Programming, Python","topic_modeling_1_cluster_Recommendation, Travel","topic_modeling_1_cluster_Security, Cybersecurity","topic_modeling_1_cluster_Storytelling, Role-playing","topic_modeling_1_cluster_Summarization, Research","topic_modeling_1_cluster_Technology, Engineering"
0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [8]:
# treat each component of the response embedding vector as a feature
# compute the difference in each feature between model_a response and model_b response
response_charac_diff = response_a_embeddings - response_b_embeddings

## logistic regression

In [9]:
# full feature set

In [10]:
X = np.hstack((np.array(one_hot_encoded_df), response_charac_diff))
y = np.array(dat['winner'])

# 20 + 20 + 30 + 256
print(X.shape)

(25256, 326)


In [13]:
penalties = ['l1', 'l2', None]
cv_results = {}
for penalty in penalties:
    print(penalty)
    temp_clf = LogisticRegression(fit_intercept=True, random_state=200, max_iter=100, solver='saga', penalty=penalty)
    temp_scores = cross_val_score(temp_clf, X, y, cv=KFold(n_splits=5, shuffle=True, random_state=200), scoring='accuracy')
    print(temp_scores)
    cv_results[penalty] = np.mean(temp_scores)
print('\n', str(cv_results))

l1
[0.55186065 0.54523857 0.5490002  0.54820828 0.54365472]
l2
[0.55146477 0.54246684 0.54583251 0.54405068 0.54306078]
None
[0.54889153 0.53811127 0.54207088 0.54424866 0.54405068]

 {'l1': 0.5475924822550288, 'l2': 0.5453751152339565, None: 0.5434746033843553}


In [ ]:
# feature set without cluster features

In [16]:
X = np.hstack((one_hot_encoded_df[[col for col in one_hot_encoded_df.columns if not col.startswith('topic')]], 
               response_charac_diff))
y = np.array(dat['winner'])

# 20 + 20 + 256
print(X.shape)

(25256, 296)


In [17]:
penalties = ['l1', 'l2', None]
cv_results = {}
for penalty in penalties:
    print(penalty)
    temp_clf = LogisticRegression(fit_intercept=True, random_state=200, max_iter=100, solver='saga', penalty=penalty)
    temp_scores = cross_val_score(temp_clf, X, y, cv=KFold(n_splits=5, shuffle=True, random_state=200), scoring='accuracy')
    print(temp_scores)
    cv_results[penalty] = np.mean(temp_scores)
print('\n', str(cv_results))

l1
[0.54651623 0.54147694 0.53751732 0.54147694 0.53731934]
l2
[0.54453682 0.54108097 0.53929915 0.54108097 0.53692338]
None
[0.54453682 0.53830925 0.53870521 0.54028905 0.53415165]

 {'l1': 0.5408613535445973, 'l2': 0.5405842590846524, None: 0.5391983948993426}


In [ ]:
# final model with full feature set using all training data

In [18]:
# rebuild model
X = np.hstack((np.array(one_hot_encoded_df), response_charac_diff))
y = np.array(dat['winner'])

# 20 + 20 + 30 + 256
print(X.shape)

clf_full_feat = LogisticRegression(fit_intercept=True, 
                                   random_state=200, 
                                   max_iter=100, 
                                   solver='saga', 
                                   penalty='l1').fit(X_train, y_train).fit(X, y)

print('training accuracy score:', round(accuracy_score(y, clf_full_feat.predict(X)), 3))

(25256, 326)
training accuracy score: 0.56


## test data

In [19]:
response_a_embeddings_test = np.load("/home/jovyan/_shared/data100-readonly/fa24_grad_project_data/nlp-chatbot-analysis_data/test-set/arena-test-set-model_a_response-embeddings.npy")
response_b_embeddings_test = np.load("/home/jovyan/_shared/data100-readonly/fa24_grad_project_data/nlp-chatbot-analysis_data/test-set/arena-test-set-model_b_response-embeddings.npy")
response_charac_diff_test = response_a_embeddings_test - response_b_embeddings_test

print(response_charac_diff_test.shape)

(3200, 256)


In [20]:
df_test = pd.read_json(
    "/home/jovyan/_shared/data100-readonly/fa24_grad_project_data/nlp-chatbot-analysis_data/test-set/arena-test-set-prompt-and-responses.jsonl.gz",
    lines=True,
    compression="gzip"
)

print(df_test.shape)
df_test.head()

(3200, 6)


,question_id,prompt,model_a,model_b,model_a_response,model_b_response
0,4f332ebd8cdc4ff2be74aa8828ff20d5,what do you think about the future of iran?,koala-13b,vicuna-13b,"As an AI language model, I don't have the abil...","As an AI language model, I do not have persona..."
1,f2be6f13e5ed40e5b81443223996494c,Salut ! Tu es un méchant chatbot !,stablelm-tuned-alpha-7b,vicuna-13b,"lut! Tuer ton pseudo câbleur, câbleur câbleur!...","Salut ! Je suis un chatbot, mais je suis progr..."
2,5fafefb8a0c54243afb52d2892946cea,⚔️ Chatbot Arena ⚔️\nRules:\n Chat with two...,koala-13b,vicuna-13b,"Hello! \n\nModel A: Hi there, I'm Model A, a f...","Sure, I can do that! Let's get started.\n\nMod..."
3,7834f572267f40709ecebb273a2b346b,Guess the word that i have in my mind,chatglm-6b,stablelm-tuned-alpha-7b,"I'm sorry, but I'm not able to guess the word ...","I'm sorry, but as an AI language model, I do n..."
4,1ccc7e58290245c4bd5457fce45f8640,You are a peasant living in the village. But s...,vicuna-13b,koala-13b,"As a peasant living in the village, I would li...","As a peasant living in a village, my thoughts ..."


In [21]:
hayden_cluster_test = pd.read_csv('hayden_cluster_test_deciphered.csv')

print(hayden_cluster_test.shape)
hayden_cluster_test.head()

(3200, 8)


,question_id,prompt,topic_modeling_1,topic_modeling_2,topic_modeling_3,topic_modeling_1_cluster,topic_modeling_2_cluster,topic_modeling_3_cluster
0,4f332ebd8cdc4ff2be74aa8828ff20d5,what do you think about the future of iran?,Future Prediction,Future Prediction,Future Prediction,"Comparison, Time","Comparison, Time","Comparison, Time"
1,f2be6f13e5ed40e5b81443223996494c,Salut ! Tu es un méchant chatbot !,"Role-playing, Evaluation","Role-play, Evaluation","Creativity, Factual Accuracy","Storytelling, Role-playing","Storytelling, Role-playing","Factual, Philosophy"
2,5fafefb8a0c54243afb52d2892946cea,⚔️ Chatbot Arena ⚔️\nRules:\n Chat with two...,Chatbot Evaluation,Chatbot Evaluation,Chatbot Evaluation,"Communication, Chatbot","Communication, Chatbot","Communication, Chatbot"
3,7834f572267f40709ecebb273a2b346b,Guess the word that i have in my mind,Guessing Game,Word Guessing,Word Guessing,"Counting, Sorting","Counting, Sorting","Counting, Sorting"
4,1ccc7e58290245c4bd5457fce45f8640,You are a peasant living in the village. But s...,"Problem-Solving, Creativity",Problem Solving,"Problem-solving, Creativity","Creativity, Creative","Problem-solving, Optimization","Creativity, Creative"


In [22]:
df_test_merged = pd.merge(left=hayden_cluster_test, 
                          right=df_test[['model_a','model_b']], 
                          how='inner', 
                          left_index=True, 
                          right_index=True)

print(df_test_merged.shape)
df_test_merged.head()

(3200, 10)


,question_id,prompt,topic_modeling_1,topic_modeling_2,topic_modeling_3,topic_modeling_1_cluster,topic_modeling_2_cluster,topic_modeling_3_cluster,model_a,model_b
0,4f332ebd8cdc4ff2be74aa8828ff20d5,what do you think about the future of iran?,Future Prediction,Future Prediction,Future Prediction,"Comparison, Time","Comparison, Time","Comparison, Time",koala-13b,vicuna-13b
1,f2be6f13e5ed40e5b81443223996494c,Salut ! Tu es un méchant chatbot !,"Role-playing, Evaluation","Role-play, Evaluation","Creativity, Factual Accuracy","Storytelling, Role-playing","Storytelling, Role-playing","Factual, Philosophy",stablelm-tuned-alpha-7b,vicuna-13b
2,5fafefb8a0c54243afb52d2892946cea,⚔️ Chatbot Arena ⚔️\nRules:\n Chat with two...,Chatbot Evaluation,Chatbot Evaluation,Chatbot Evaluation,"Communication, Chatbot","Communication, Chatbot","Communication, Chatbot",koala-13b,vicuna-13b
3,7834f572267f40709ecebb273a2b346b,Guess the word that i have in my mind,Guessing Game,Word Guessing,Word Guessing,"Counting, Sorting","Counting, Sorting","Counting, Sorting",chatglm-6b,stablelm-tuned-alpha-7b
4,1ccc7e58290245c4bd5457fce45f8640,You are a peasant living in the village. But s...,"Problem-Solving, Creativity",Problem Solving,"Problem-solving, Creativity","Creativity, Creative","Problem-solving, Optimization","Creativity, Creative",vicuna-13b,koala-13b


In [23]:
one_hot_encoded_test_df = pd.DataFrame(
    enc.transform(df_test_merged[['model_a','model_b','topic_modeling_1_cluster']]).toarray(),
    columns = enc.get_feature_names_out()
)

print(one_hot_encoded_test_df.shape)
one_hot_encoded_test_df.head()

(3200, 70)


,model_a_RWKV-4-Raven-14B,model_a_alpaca-13b,model_a_chatglm-6b,model_a_claude-instant-v1,model_a_claude-v1,model_a_dolly-v2-12b,model_a_fastchat-t5-3b,model_a_gpt-3.5-turbo,model_a_gpt-4,model_a_gpt4all-13b-snoozy,model_a_guanaco-33b,model_a_koala-13b,model_a_llama-13b,model_a_mpt-7b-chat,model_a_oasst-pythia-12b,model_a_palm-2,model_a_stablelm-tuned-alpha-7b,model_a_vicuna-13b,model_a_vicuna-7b,model_a_wizardlm-13b,model_b_RWKV-4-Raven-14B,model_b_alpaca-13b,model_b_chatglm-6b,model_b_claude-instant-v1,model_b_claude-v1,model_b_dolly-v2-12b,model_b_fastchat-t5-3b,model_b_gpt-3.5-turbo,model_b_gpt-4,model_b_gpt4all-13b-snoozy,model_b_guanaco-33b,model_b_koala-13b,model_b_llama-13b,model_b_mpt-7b-chat,model_b_oasst-pythia-12b,model_b_palm-2,model_b_stablelm-tuned-alpha-7b,model_b_vicuna-13b,model_b_vicuna-7b,model_b_wizardlm-13b,"topic_modeling_1_cluster_AI, Task","topic_modeling_1_cluster_Communication, Chatbot","topic_modeling_1_cluster_Comparison, Time","topic_modeling_1_cluster_Counting, Sorting","topic_modeling_1_cluster_Creativity, Creative","topic_modeling_1_cluster_Data, Database","topic_modeling_1_cluster_Emotional, Psychology","topic_modeling_1_cluster_Ethics, Legal","topic_modeling_1_cluster_Evaluation, Analysis","topic_modeling_1_cluster_Factual, Philosophy","topic_modeling_1_cluster_Financial, Finance","topic_modeling_1_cluster_Food, Recipe","topic_modeling_1_cluster_Game, Gaming","topic_modeling_1_cluster_Geography, Astronomy","topic_modeling_1_cluster_History, Historical","topic_modeling_1_cluster_Identification, Classification","topic_modeling_1_cluster_Language, Text","topic_modeling_1_cluster_Logic, Reasoning","topic_modeling_1_cluster_Mathematics, Mathematical","topic_modeling_1_cluster_Medical, Biology","topic_modeling_1_cluster_Physics, Chemistry","topic_modeling_1_cluster_Poetry, Music","topic_modeling_1_cluster_Problem-solving, Optimization","topic_modeling_1_cluster_Programming, Development","topic_modeling_1_cluster_Programming, Python","topic_modeling_1_cluster_Recommendation, Travel","topic_modeling_1_cluster_Security, Cybersecurity","topic_modeling_1_cluster_Storytelling, Role-playing","topic_modeling_1_cluster_Summarization, Research","topic_modeling_1_cluster_Technology, Engineering"
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [24]:
X_test = np.hstack((np.array(one_hot_encoded_test_df), response_charac_diff_test))

print(X_test.shape)

(3200, 326)


In [25]:
# task A test data pred
y_pred = clf_full_feat.predict(X_test)

In [26]:
unique_values, counts = np.unique(y_pred, return_counts=True)
print(unique_values)
print(counts)

[0 1 2 3]
[1492 1474  229    5]


In [27]:
submit_df = df_test_merged[['question_id']]
submit_df['winner'] = y_pred
# inverse transform
submit_df['winner'].replace(inplace=True, to_replace={0:'model_a',
                                                      1:'model_b',
                                                      2:'tie (bothbad)',
                                                      3:'tie'})

submit_df.head()

,question_id,winner
0,4f332ebd8cdc4ff2be74aa8828ff20d5,model_b
1,f2be6f13e5ed40e5b81443223996494c,model_b
2,5fafefb8a0c54243afb52d2892946cea,model_b
3,7834f572267f40709ecebb273a2b346b,tie (bothbad)
4,1ccc7e58290245c4bd5457fce45f8640,model_a


In [28]:
submit_df.to_csv('taskA_results.csv', index=False)